### Data Ingestion To Vector DB Pipeline

In [2]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
# from langchain.textsplitters import RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter


d:\Projects\traditionalRag\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data/pdf")

Found 2 PDF files to process

Processing: Lecture 10.pdf
  ✓ Loaded 75 pages

Processing: Lecture 12.pdf
  ✓ Loaded 55 pages

Total documents loaded: 130


In [4]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs

In [5]:
chunked_documents = split_documents(all_pdf_documents)  

Split 130 documents into 163 chunks

Example chunk:
Content: 1
Back-End Development
Web Servers,
Back-end Abstractions,
Building APIs
1...
Metadata: {'producer': 'Microsoft® PowerPoint® for Microsoft 365', 'creator': 'Microsoft® PowerPoint® for Microsoft 365', 'creationdate': '2024-12-04T00:34:22+05:00', 'title': '', 'author': 'DELL', 'moddate': '2024-12-04T00:34:22+05:00', 'source': '..\\data\\pdf\\Lecture 10.pdf', 'total_pages': 75, 'page': 0, 'page_label': '1', 'source_file': 'Lecture 10.pdf', 'file_type': 'pdf'}


In [6]:
import numpy as np
import chromadb
from sentence_transformers import SentenceTransformer
import uuid
from chromadb.config import Settings
from typing import List, Dict, Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [7]:
class EmbeddingManager:
    """Handles Document Embedding Using Sentence Transformer"""
    
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Initialize Embedding Manager
        
        Args: 
            Hugging face model name for sentence transformer (default: "all-MiniLM-L6-v2")
        """
        self.model_name = model_name
        self.model = None
        self.load_model()

    def load_model(self):
        """Load the sentence transformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}...")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Dimensions: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model: {e}")
            raise

    def generateEmbeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts
        
        Args:
            texts: List of strings to embed
        Returns:
            Numpy array of embeddings
        """
        if not self.model:
            raise ValueError("Model not loaded")
        try:
            print(f"Generating embeddings for {len(texts)} texts...")
            embeddings = self.model.encode(texts, show_progress_bar=True)
            print(f"Generated embeddings with shape: {embeddings.shape}")
            return np.array(embeddings)
        except Exception as e:
            print(f"Error generating embeddings: {e}")
            raise


In [8]:
embedding_manager = EmbeddingManager()

Loading embedding model: all-MiniLM-L6-v2...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 625.67it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Dimensions: 384


### VectorDatabase


In [16]:
class VectorStore:
    def __init__(self, collection_name: str = "pdf_chunks", persistent_directory: str = "../data/chroma_db"):
        """Initialize the vector store (ChromaDB)"""
        self.collection_name = collection_name
        self.persistent_directory = persistent_directory
        self.client = None
        self.collection = None
        self._connect_to_db()
    
    def _connect_to_db(self):
        """Create persistent chroma db client and collection"""
        try:
            os.makedirs(self.persistent_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persistent_directory)
            
            # Get or create collection
            self.collection = self.client.get_or_create_collection(name=self.collection_name)
            print(f"Vector store connected: {self.collection_name}")
            print(f"Existing collections: {self.collection.count()}")
        except Exception as e:
            print(f"Error connecting to ChromaDB: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

              # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise



In [17]:
store = VectorStore()

Vector store connected: pdf_chunks
Existing collections: 0


In [11]:
chunked_documents


[Document(metadata={'producer': 'Microsoft® PowerPoint® for Microsoft 365', 'creator': 'Microsoft® PowerPoint® for Microsoft 365', 'creationdate': '2024-12-04T00:34:22+05:00', 'title': '', 'author': 'DELL', 'moddate': '2024-12-04T00:34:22+05:00', 'source': '..\\data\\pdf\\Lecture 10.pdf', 'total_pages': 75, 'page': 0, 'page_label': '1', 'source_file': 'Lecture 10.pdf', 'file_type': 'pdf'}, page_content='1\nBack-End Development\nWeb Servers,\nBack-end Abstractions,\nBuilding APIs\n1'),
 Document(metadata={'producer': 'Microsoft® PowerPoint® for Microsoft 365', 'creator': 'Microsoft® PowerPoint® for Microsoft 365', 'creationdate': '2024-12-04T00:34:22+05:00', 'title': '', 'author': 'DELL', 'moddate': '2024-12-04T00:34:22+05:00', 'source': '..\\data\\pdf\\Lecture 10.pdf', 'total_pages': 75, 'page': 1, 'page_label': '2', 'source_file': 'Lecture 10.pdf', 'file_type': 'pdf'}, page_content='Lec 10: Web Servers\n2'),
 Document(metadata={'producer': 'Microsoft® PowerPoint® for Microsoft 365', '

In [12]:
texts=[doc.page_content for doc in chunked_documents]
texts

['1\nBack-End Development\nWeb Servers,\nBack-end Abstractions,\nBuilding APIs\n1',
 'Lec 10: Web Servers\n2',
 'What is a web server?\nA web server is both a type of software and a piece of hardware \nthat stores, processes, and delivers web pages to users over the \ninternet.\nSteps:\n• User types an address "www.example.com, " or click a link\n• Computer (called the "client") sends a request to the web server\n• Server then sends the right information back to the client.\n• A web server is both a type of software and a piece of hardware that stores, \nprocesses, and delivers web pages to users over the internet. When you type in a \nwebsite address, like "www.example.com," or click a link, your computer (called \nthe "client") sends a request to the web server, which then sends the right \ninformation back to your computer.\n• Think of a web server as a waiter in a restaurant:\n• When you, the customer, ask for a dish (like clicking a link), the waiter (the web \nserver) takes your 

In [13]:
embeddings=embedding_manager.generateEmbeddings(texts)

Generating embeddings for 163 texts...


Batches: 100%|██████████| 6/6 [00:25<00:00,  4.23s/it]

Generated embeddings with shape: (163, 384)


In [18]:
store.add_documents(chunked_documents,embeddings)

Adding 163 documents to vector store...
Successfully added 163 documents to vector store
Total documents in collection: 163


### RAG Retriever

In [26]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever
        
        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generateEmbeddings([query])[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever=RAGRetriever(store,embedding_manager)

In [27]:
rag_retriever

In [28]:
rag_retriever.retrieve("What is Reverse Proxy", top_k=3, score_threshold=0.5)

Retrieving documents for query: 'What is Reverse Proxy'
Top K: 3, Score threshold: 0.5
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  1.30it/s]


Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)


[{'id': 'doc_bec72c58_51',
  'content': 'Reverse Proxy\n• A reverse proxy is a type of server that sits in front of web \nservers and directs client requests to the appropriate \nbackend server. It provides an extra layer of security and \nhelps in load balancing as well.\n• Benefits:\n• Security: Protects the identity and location of the backend server.\n• Efficiency: Redirects requests to different servers based on load.\n• Example: NGINX can act as a reverse proxy. If a site has \nmultiple servers, NGINX can manage which server a client is \ndirected to.\n32',
  'metadata': {'page_label': '38',
   'page': 37,
   'moddate': '2024-12-04T00:34:22+05:00',
   'content_length': 515,
   'source': '..\\data\\pdf\\Lecture 10.pdf',
   'title': '',
   'source_file': 'Lecture 10.pdf',
   'file_type': 'pdf',
   'total_pages': 75,
   'creationdate': '2024-12-04T00:34:22+05:00',
   'producer': 'Microsoft® PowerPoint® for Microsoft 365',
   'author': 'DELL',
   'doc_index': 51,
   'creator': 'Micro

In [29]:
import os
from dotenv import load_dotenv
load_dotenv()

print(os.getenv("GROQ_API_KEY"))

None


In [46]:
from dotenv import load_dotenv
load_dotenv()

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage

In [50]:
class GroqLLM:
    def __init__(self, model_name: str = "gemma2-9b-it", api_key: str =None):
        """
        Initialize Groq LLM
        
        Args:
            model_name: Groq model name (qwen2-72b-instruct, llama3-70b-8192, etc.)
            api_key: Groq API key (or set GROQ_API_KEY environment variable)
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")
        
        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")
        
        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )
        
        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        
        Args:
            query: User question
            context: Retrieved document context
            max_length: Maximum response length
            
        Returns:
            Generated response string
        """
        
        # Create prompt template
        prompt_template = ChatPromptTemplate.from_messages([
            ("system", "You are a helpful AI assistant. Use the following context to answer the question accurately and concisely."),
            ("human", "Context:\n{context}\n\nQuestion: {question}\n\nAnswer:")
        ])
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content 
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)
        
        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
            
        except Exception as e:
            return f"Error generating response: {str(e)}"
        
    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        
        Args:
            query: User question
            context: Retrieved context
            
        Returns:
            Generated response
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""
        
        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"

In [51]:
# Initialize Groq LLM (loads GROQ_API_KEY from .env file)
GROQ_API_KEY = os.environ.get("GROQ_API_KEY")

try:
    groq_llm = GroqLLM(api_key=GROQ_API_KEY)
    print("Groq LLM initialized successfully!")
    print(f"Using model: {groq_llm.model_name}")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set GROQ_API_KEY in your .env file or as an environment variable.")
    groq_llm = None

Initialized Groq LLM with model: gemma2-9b-it
Groq LLM initialized successfully!
Using model: gemma2-9b-it


TypeError: ChatPromptTemplate.__init__() missing 1 required positional argument: 'messages'

In [ ]:
# Test Groq LLM with RAG
if groq_llm is not None:
    # Test query
    test_query = "What are the main topics discussed in the documents?"
    
    print("=" * 80)
    print("Testing Groq LLM with RAG Pipeline")
    print("=" * 80)
    print(f"\nTest Query: {test_query}\n")
    
    # Step 1: Retrieve relevant documents
    print("Step 1: Retrieving relevant documents...")
    retrieved_docs = rag_retriever.retrieve(test_query, top_k=3, score_threshold=0.0)
    print(f"Retrieved {len(retrieved_docs)} documents\n")
    
    # Step 2: Combine retrieved documents as context
    if retrieved_docs:
        context = "\n\n".join([
            f"Document {doc['rank']}: {doc['content'][:500]}..." 
            for doc in retrieved_docs
        ])
        
        print("Step 2: Generating response using Groq LLM...")
        print("-" * 80)
        
        try:
            # Call Groq LLM with context
            response = groq_llm.generate_response(test_query, context)
            
            print(f"\nResponse from Groq LLM:\n")
            print(response)
            print("-" * 80)
            print("\n✓ Groq LLM test successful!")
            
        except Exception as e:
            print(f"✗ Error calling Groq LLM: {e}")
    else:
        print("No documents retrieved. Cannot test LLM generation.")
else:
    print("Groq LLM is not initialized. Cannot run test.")